In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l6.8 Ablation harness

Claims about architecture mean nothing without controlled comparison. An ablation
harness trains each variant under identical conditions and tabulates the deltas.
This produces the modernized PocketLM and the evidence for its choices.

In [ ]:
from pocketlm import (CharTokenizer, PocketLM, PocketLMConfig, init_weights,
                      make_batch, estimate_loss)

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)

def run(**overrides):
    torch.manual_seed(0)
    cfg = PocketLMConfig(vocab_size=tok.vocab_size, block_size=32, n_layer=2,
                         n_head=4, n_embd=64, **overrides)
    model = init_weights(PocketLM(cfg))
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    g = torch.Generator().manual_seed(0)
    steps = 120 if os.environ.get("POCKETLM_CI") else 400
    for _ in range(steps):
        x, y = make_batch(data, 32, 16, generator=g)
        _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    return estimate_loss(model, data, 32, 16, iters=10, generator=g)

variants = {
    "baseline (LN+GELU+learned)": {},
    "+ rmsnorm": dict(norm="rmsnorm"),
    "+ swiglu": dict(mlp="swiglu"),
    "modern (rms+swiglu+rope)": dict(norm="rmsnorm", mlp="swiglu", pos="rope"),
}
results = {name: run(**cfg) for name, cfg in variants.items()}
for name, loss in results.items():
    print(f"{name:<28} val loss {loss:.3f}")

Now the choices are evidence-based, not folklore. A real harness would average over
seeds and report variance; the discipline, identical conditions per variant, is the
same at any scale.

In [ ]:
import math
assert len(results) == 4
assert all(loss < math.log(tok.vocab_size) for loss in results.values())